In [ ]:
import sys, os, random
sys.path.append('/content/genAI-project-24f1002692')

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, TrainingArguments, Trainer, default_data_collator

from src.dataset.data_loader import load_competition_data
from src.models.LoRA_MC import (
    OPTIONS, LABEL_TO_ID, ID_TO_LABEL, DEFAULT_MODEL_NAME,
    build_hf_mc_dataset, build_lora_mc_model, mc_compute_metrics, MC_LoRA_Predictor,
)
from src.inference.LoRA_MC_inference import run_lora_mc_predictions, save_submission

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
# Load competition data, encode answer letters (A-E) as numeric labels
train_dataset, test_dataset, sample_submission = load_competition_data()

train_df = train_dataset.copy()
train_df['label'] = train_df['answer'].map(LABEL_TO_ID)
test_df = test_dataset.copy()

print("Label mapping:", LABEL_TO_ID)
print(train_df[['answer', 'label']].head())

In [ ]:
# Prompt-level train/val split (prevents the same question leaking across both sets)
prompt_answer = train_df.groupby('prompt')['answer'].first()
unique_prompts = prompt_answer.index.values

train_prompts, val_prompts = train_test_split(
    unique_prompts, test_size=0.1, random_state=SEED, stratify=prompt_answer.values
)

train_split = train_df[train_df['prompt'].isin(train_prompts)].reset_index(drop=True)
val_split   = train_df[train_df['prompt'].isin(val_prompts)].reset_index(drop=True)

overlap = set(train_split['prompt']) & set(val_split['prompt'])
print(f"Train rows: {len(train_split)}  |  Val rows: {len(val_split)}  |  Prompt overlap: {len(overlap)}")

In [ ]:
# Tokenizer + LoRA-wrapped bert-base-uncased (r=8, alpha=16, target=[query,value])
MC_MAX_LENGTH = 128

MC_tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL_NAME)
MC_model = build_lora_mc_model(DEFAULT_MODEL_NAME)
MC_model.to(device)

hf_mc_train = build_hf_mc_dataset(train_split, MC_tokenizer, max_length=MC_MAX_LENGTH)
hf_mc_val   = build_hf_mc_dataset(val_split, MC_tokenizer, max_length=MC_MAX_LENGTH)

print(hf_mc_train)
print(hf_mc_val)

In [ ]:
# Fine-tune with HF Trainer (tracks accuracy + MAP@3, keeps best-MAP@3 checkpoint)
MC_training_args = TrainingArguments(
    output_dir="./m4_lora_MC_model",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=18,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    logging_steps=20,
    report_to=["wandb"],
    fp16=torch.cuda.is_available(),
)

MC_trainer = Trainer(
    model=MC_model,
    args=MC_training_args,
    train_dataset=hf_mc_train,
    eval_dataset=hf_mc_val,
    data_collator=default_data_collator,
    compute_metrics=mc_compute_metrics,
)

MC_train_result = MC_trainer.train()
print(MC_train_result)

In [ ]:
# Evaluate + save the best LoRA adapter checkpoint
MC_eval_metrics = MC_trainer.evaluate()
print("Final validation metrics:", MC_eval_metrics)

MC_trainer.save_model("./m4_lora_mc_model_best")
print("Best LoRA multiple-choice model saved to ./m4_lora_mc_model_best")

In [ ]:
# Load the saved LoRA adapter (merged into the base model), predict on test.csv, save submission
predictor = MC_LoRA_Predictor(
    lora_checkpoint_dir="./m4_lora_mc_model_best",
    base_model_name=DEFAULT_MODEL_NAME,
    max_length=MC_MAX_LENGTH,
    device=device,
)

submission = run_lora_mc_predictions(predictor, test_df, id_col="id", batch_size=32)
save_submission(submission, out_path="submission_lora_mc.csv")
submission.head(10)